In [3]:
import json

# This function converts Google Vision API output to COCO format.
def convert_to_coco_format(vision_response, image_id, image_filename, width, height):
    coco_format = {
        "images": [
            {
                "id": image_id,
                "width": width,
                "height": height,
                "file_name": image_filename,
            }
        ],
        "annotations": [],
        "categories": [
            {
                "id": 1,
                "name": "text",
                "supercategory": "none",
            }
        ]
    }

    annotation_id = 1
    for page in vision_response.full_text_annotation.pages:
        for block in page.blocks:
            for paragraph in block.paragraphs:
                for word in paragraph.words:
                    # Get bounding box for each word
                    vertices = word.bounding_box.vertices
                    xmin = min(vertex.x for vertex in vertices if vertex.x)
                    ymin = min(vertex.y for vertex in vertices if vertex.y)
                    xmax = max(vertex.x for vertex in vertices if vertex.x)
                    ymax = max(vertex.y for vertex in vertices if vertex.y)
                    width_box = xmax - xmin
                    height_box = ymax - ymin

                    # Add annotation
                    coco_format['annotations'].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": 1,  # Text category
                        "bbox": [xmin, ymin, width_box, height_box],
                        "area": width_box * height_box,
                        "iscrowd": 0
                    })
                    annotation_id += 1

    return coco_format

Object Detection

In [2]:
from google.cloud import vision
from google.oauth2 import service_account

def find_book_object(key_path,image_path):
    '''
    Detects a book object in an image using Google Cloud Vision API.

    Parameters:
    service_account_path (str): Path to the Google Cloud service account key file.
    image_path (str): Path to the image file.

    Returns:
    The detected book object or None if no book is found.
    '''
    # Provide the path to your service account key file
    credentials = service_account.Credentials.from_service_account_file(key_path)

    # Create a client
    client = vision.ImageAnnotatorClient(credentials=credentials)

    # Load your image into memory
    with open(image_path, 'rb') as image_file:
        content = image_file.read()
    image = vision.Image(content=content)

    # Perform object detection
    response = client.object_localization(image=image)
    objects = response.localized_object_annotations

     # Perform object detection
    response = client.object_localization(image=image)
    objects = response.localized_object_annotations

    # Search for a book object
    for object_ in objects:
        if object_.name.lower() == 'book':
            return object_

    return None

import boto3

def detect_book_with_rekognition(image_path):
    client = boto3.client('rekognition', region_name='us-east-2')
    with open(image_path, 'rb') as image:
        response = client.detect_labels(Image={'Bytes': image.read()})

    for label in response['Labels']:
        if label['Name'].lower() == 'book':
            return label['Instances'][0]['BoundingBox'] if label['Instances'] else None
    return None

from PIL import Image
import os


from PIL import Image
import os

def crop_book_image(book_object, image_path, output_folder):
    """
    Crops the book object from an image and saves it.

    Parameters:
    book_object: The book object returned from a detection service.
    image_path (str): Path to the image file.
    output_folder (str): Path to the folder where the cropped image will be saved.

    Returns:
    The path to the saved cropped image or None if no book object is provided.
    """
    if not book_object:
        return None

    # Load the original image with PIL
    pil_image = Image.open(image_path)

    # Get the dimensions of the original image
    width, height = pil_image.size

    # Determine the source of book_object and calculate the bounding box
    if 'bounding_poly' in book_object and hasattr(book_object.bounding_poly, 'normalized_vertices'):  # GCV format
        vertices = book_object.bounding_poly.normalized_vertices
        xmin = min([v.x for v in vertices]) * width
        ymin = min([v.y for v in vertices]) * height
        xmax = max([v.x for v in vertices]) * width
        ymax = max([v.y for v in vertices]) * height
    else:  # AWS Rekognition format
        bbox = book_object
        xmin = bbox['Left'] * width
        ymin = bbox['Top'] * height
        xmax = (bbox['Left'] + bbox['Width']) * width
        ymax = (bbox['Top'] + bbox['Height']) * height

    # Crop the image
    cropped_image = pil_image.crop((xmin, ymin, xmax, ymax))

    # Save the cropped image
    output_path = os.path.join(output_folder, 'cropped_book.jpg')
    cropped_image.save(output_path)

    return output_path

import numpy as np

def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w / 2, h / 2)

    # Calculate the rotation matrix
    M = cv2.getRotationMatrix2D(center, angle, 1.0)

    # Calculate the sine and cosine of rotation angle
    abs_cos = abs(M[0, 0])
    abs_sin = abs(M[0, 1])

    # Calculate new width and height
    new_w = int(h * abs_sin + w * abs_cos)
    new_h = int(h * abs_cos + w * abs_sin)

    # Adjust the rotation matrix to consider the translation
    M[0, 2] += new_w / 2 - center[0]
    M[1, 2] += new_h / 2 - center[1]

    # Perform the rotation
    rotated = cv2.warpAffine(image, M, (new_w, new_h))
    return rotated

def find_dividing_line(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        image_center = image.shape[1] / 2  # Width, since the line is vertical
        closest_line = min(lines, key=lambda line: abs((line[0][0] + line[0][2]) / 2 - image_center))
        dividing_x = int((closest_line[0][0] + closest_line[0][2]) / 2)
        return dividing_x
    else:
        return image.shape[1] // 2

def split_page(image, dividing_line, overlap=5):
    """
    Splits the image into two parts with a slight overlap.
    :param image: The image to split.
    :param dividing_line: The x-coordinate (for vertical split) of the dividing line.
    :param overlap: The number of pixels to overlap the two parts.
    :return: A tuple containing the left and right parts of the image.
    """
    left_part = image[:, :dividing_line + overlap]
    right_part = image[:, dividing_line - overlap:]
    return left_part, right_part
def save_image(cv_image, filename):
    pil_image = Image.fromarray(cv2.cvtColor(cv_image, cv2.COLOR_BGR2RGB))
    pil_image.save(filename)

import matplotlib.pyplot as plt

def display_images(images):
    plt.figure(figsize=(10, 10))

    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off')

    plt.show()


In [6]:
Year=1942
import os
User = os.environ.get('USERNAME')  # For Windows
ObjectDetectFail=[]

for Page in range(8,len(os.listdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/' + str(Year) + '/HQ')),1):
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/HQ/Page'+f'{Page:03d}'+'.jpg'
    path='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/'
    key_path = path+'Tokyo_Jobs/Environment/GoogleVision/practice-302516-01e0f7245b03.json'

    ##Get Book Object
    book_object = find_book_object(key_path, image_path)
    if book_object:
        print(f'Object Name: {book_object.name}')
        print('Confidence: {}'.format(book_object.score))
        print('Normalized bounding polygon vertices:')
        for vertex in book_object.bounding_poly.normalized_vertices:
            print(' - ({}, {})'.format(vertex.x, vertex.y))
    elif book_object:
        print('No book object found in the image by GCV. Trying AWS Rekognition...')
        book_object = detect_book_with_rekognition(image_path)
        if book_object:
            print('Book object found with AWS Rekognition')
        else:
            print('No book object found in the image by GCV nor Rekognition.')
            ObjectDetectFail.append(Page)
    

    save_folder='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book'
    cropped_image_path =crop_book_image(book_object, image_path, save_folder)

    if cropped_image_path:
        print(f'Cropped image saved to: {cropped_image_path}')
    else:
        print('No book object found or could not crop the image.')

    ## Seperate Directory
    import cv2
    import numpy as np
    from PIL import Image

    # Load the image using OpenCV
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book/cropped_book.jpg'
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray, threshold1=50, threshold2=150)

    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        image_center = image.shape[1] / 2
        closest_line = min(lines, key=lambda line: abs((line[0][0] + line[0][2]) / 2 - image_center))
        spine_x = int((closest_line[0][0] + closest_line[0][2]) / 2)
    else:
        spine_x = image.shape[1] // 2  # Fallback to the middle if no line is found

    left_page = image[:, :spine_x]
    right_page = image[:, spine_x:]

    # Assuming left_page and right_page are the already split pages
    left_page_rotated = rotate_image(left_page, 90)
    right_page_rotated = rotate_image(right_page, 90)


    dividing_line_left = find_dividing_line(left_page_rotated)
    dividing_line_right = find_dividing_line(right_page_rotated)

    left_top, left_bottom = split_page(left_page_rotated, dividing_line_left)
    left_top, left_bottom = rotate_image(left_top, -90),rotate_image(left_bottom, -90)
    right_top, right_bottom = split_page(right_page_rotated, dividing_line_right)
    right_top, right_bottom = rotate_image(right_top, -90),rotate_image(right_bottom, -90)

    if not os.path.exists('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img'):
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}')
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img')
    save_image(left_top, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/left_top.jpg')
    save_image(left_bottom, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/left_bottom.jpg')
    save_image(right_top, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/right_top.jpg')
    save_image(right_bottom, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/right_bottom.jpg')


No book object found or could not crop the image.
Object Name: Book
Confidence: 0.6230558753013611
Normalized bounding polygon vertices:
 - (0.044692859053611755, 0.09522777795791626)
 - (0.9410567879676819, 0.09522777795791626)
 - (0.9410567879676819, 0.837033748626709)
 - (0.044692859053611755, 0.837033748626709)
Cropped image saved to: C:/Users/keitaro2/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1942/Book\cropped_book.jpg
Object Name: Book
Confidence: 0.7299599647521973
Normalized bounding polygon vertices:
 - (0.048303309828042984, 0.0920606181025505)
 - (0.9396721720695496, 0.0920606181025505)
 - (0.9396721720695496, 0.8396434783935547)
 - (0.048303309828042984, 0.8396434783935547)
Cropped image saved to: C:/Users/keitaro2/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1942/Book\cropped_book.jpg
Object Name: Book
Confidence: 0.6387485265731812
Normalized bounding polygon vertices:
 - (0.046148598194122314, 0.09773387759923935)
 - (0.

In [12]:
Year=1935
import os
User = os.environ.get('USERNAME')  # For Windows

for Page in range(1,len(os.listdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/' +str(Year)+ '/HQ')),1):
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/HQ/Page'+f'{Page:03d}'+'.jpg'
    path='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/'
    key_path = path+'Tokyo_Jobs/Environment/GoogleVision/practice-302516-01e0f7245b03.json'

    ##Get Book Object
    book_object = find_book_object(key_path, image_path)
    if book_object:
        print(f'Object Name: {book_object.name}')
        print('Confidence: {}'.format(book_object.score))
        print('Normalized bounding polygon vertices:')
        for vertex in book_object.bounding_poly.normalized_vertices:
            print(' - ({}, {})'.format(vertex.x, vertex.y))
    elif book_object:
        print('No book object found in the image by GCV. Trying AWS Rekognition...')
        book_object = detect_book_with_rekognition(image_path)
        if book_object:
            print('Book object found with AWS Rekognition')
        else:
            print('No book object found in the image by GCV nor Rekognition.')
            ObjectDetectFail.append(Page)

    save_folder='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book'
    cropped_image_path =crop_book_image(book_object, image_path, save_folder)

    if cropped_image_path:
        print(f'Cropped image saved to: {cropped_image_path}')
    else:
        print('No book object found or could not crop the image.')
        continue

    ## Seperate Directory
    import cv2
    import numpy as np
    from PIL import Image

    # Load the image using OpenCV
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book/cropped_book.jpg'
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray, threshold1=50, threshold2=150)

    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        image_center = image.shape[1] / 2
        closest_line = min(lines, key=lambda line: abs((line[0][0] + line[0][2]) / 2 - image_center))
        spine_x = int((closest_line[0][0] + closest_line[0][2]) / 2)
    else:
        spine_x = image.shape[1] // 2  # Fallback to the middle if no line is found

    left_page = image[:, :spine_x]
    right_page = image[:, spine_x:]

    if not os.path.exists('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page'):
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}')
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page')
    save_image(left_page, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page/left_page.jpg')
    save_image(right_page, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page/right_page.jpg')


No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.
No book object found or could not crop the image.


In [11]:
Year=1950
import os
User = os.environ.get('USERNAME')  # For Windows


for Page in range(1,len(os.listdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/' + str(Year) + '/HQ')),1):
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/HQ/Page'+f'{Page:03d}'+'.jpg'
    path='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/'
    key_path = path+'Tokyo_Jobs/Environment/GoogleVision/practice-302516-01e0f7245b03.json'

    ##Get Book Object
    book_object = find_book_object(key_path, image_path)

    if book_object:
        print(f'Object Name: {book_object.name}')
        print('Confidence: {}'.format(book_object.score))
        print('Normalized bounding polygon vertices:')
        for vertex in book_object.bounding_poly.normalized_vertices:
            print(' - ({}, {})'.format(vertex.x, vertex.y))
    else:
        print('No book object found in the image.')

    save_folder='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book'
    cropped_image_path =crop_book_image(book_object, image_path, save_folder)

    if cropped_image_path:
        print(f'Cropped image saved to: {cropped_image_path}')
    else:
        print('No book object found or could not crop the image.')
        continue


    ## Seperate Directory
    import cv2
    import numpy as np
    from PIL import Image

    # Load the image using OpenCV
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book/cropped_book.jpg'
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray, threshold1=50, threshold2=150)

    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        image_center = image.shape[1] / 2
        closest_line = min(lines, key=lambda line: abs((line[0][0] + line[0][2]) / 2 - image_center))
        spine_x = int((closest_line[0][0] + closest_line[0][2]) / 2)
    else:
        spine_x = image.shape[1] // 2  # Fallback to the middle if no line is found

    left_page = image[:, :spine_x]
    right_page = image[:, spine_x:]

    # Assuming left_page and right_page are the already split pages
    left_page_rotated = rotate_image(left_page, 90)
    right_page_rotated = rotate_image(right_page, 90)


    dividing_line_left = find_dividing_line(left_page_rotated)
    dividing_line_right = find_dividing_line(right_page_rotated)

    left_top, left_bottom = split_page(left_page_rotated, dividing_line_left)
    left_top, left_bottom = rotate_image(left_top, -90),rotate_image(left_bottom, -90)
    right_top, right_bottom = split_page(right_page_rotated, dividing_line_right)
    right_top, right_bottom = rotate_image(right_top, -90),rotate_image(right_bottom, -90)

    if not os.path.exists('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img'):
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}')
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img')
    save_image(left_top, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/left_top.jpg')
    save_image(left_bottom, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/left_bottom.jpg')
    save_image(right_top, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/right_top.jpg')
    save_image(right_bottom, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/img/right_bottom.jpg')


KeyboardInterrupt: 

In [15]:
Year=1950
import os
User = os.environ.get('USERNAME')  # For Windows


for Page in range(10,len(os.listdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/' + str(Year) + '/HQ')),10):
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/HQ/Page'+f'{Page:03d}'+'.jpg'
    path='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/'
    key_path = path+'Tokyo_Jobs/Environment/GoogleVision/practice-302516-01e0f7245b03.json'

    ##Get Book Object
    book_object = find_book_object(key_path, image_path)

    if book_object:
        print(f'Object Name: {book_object.name}')
        print('Confidence: {}'.format(book_object.score))
        print('Normalized bounding polygon vertices:')
        for vertex in book_object.bounding_poly.normalized_vertices:
            print(' - ({}, {})'.format(vertex.x, vertex.y))
    else:
        print('No book object found in the image.')

    save_folder='C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book'
    cropped_image_path =crop_book_image(book_object, image_path, save_folder)

    if cropped_image_path:
        print(f'Cropped image saved to: {cropped_image_path}')
    else:
        print('No book object found or could not crop the image.')
        continue


    ## Seperate Directory
    import cv2
    import numpy as np
    from PIL import Image

    # Load the image using OpenCV
    image_path = 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Book/cropped_book.jpg'
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray, threshold1=50, threshold2=150)

    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        image_center = image.shape[1] / 2
        closest_line = min(lines, key=lambda line: abs((line[0][0] + line[0][2]) / 2 - image_center))
        spine_x = int((closest_line[0][0] + closest_line[0][2]) / 2)
    else:
        spine_x = image.shape[1] // 2  # Fallback to the middle if no line is found

    left_page = image[:, :spine_x]
    right_page = image[:, spine_x:]

    # Assuming left_page and right_page are the already split pages
    left_page_rotated = rotate_image(left_page, 90)
    right_page_rotated = rotate_image(right_page, 90)


    dividing_line_left = find_dividing_line(left_page_rotated)
    dividing_line_right = find_dividing_line(right_page_rotated)

    left_top, left_bottom = split_page(left_page_rotated, dividing_line_left)
    left_top, left_bottom = rotate_image(left_top, -90),rotate_image(left_bottom, -90)
    right_top, right_bottom = split_page(right_page_rotated, dividing_line_right)
    right_top, right_bottom = rotate_image(right_top, -90),rotate_image(right_bottom, -90)

    if not os.path.exists('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page'):
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}')
        os.mkdir('C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page')
    save_image(left_top, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page/left_top.jpg')
    save_image(left_bottom, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page/left_bottom.jpg')
    save_image(right_top, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page/right_top.jpg')
    save_image(right_bottom, 'C:/Users/'+User+'/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/'+str(Year)+'/Page'+f'{Page:03d}'+'/Page/right_bottom.jpg')


No book object found in the image.
No book object found or could not crop the image.
No book object found in the image.
No book object found or could not crop the image.
No book object found in the image.
No book object found or could not crop the image.
No book object found in the image.
No book object found or could not crop the image.
Object Name: Book
Confidence: 0.8063350319862366
Normalized bounding polygon vertices:
 - (0.04921698942780495, 0.0262512918561697)
 - (0.9442878365516663, 0.0262512918561697)
 - (0.9442878365516663, 0.9820507168769836)
 - (0.04921698942780495, 0.9820507168769836)
Cropped image saved to: C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1950/Book\cropped_book.jpg
Object Name: Book
Confidence: 0.7092992663383484
Normalized bounding polygon vertices:
 - (0.030689872801303864, 0.031447943300008774)
 - (0.9648057818412781, 0.031447943300008774)
 - (0.9648057818412781, 0.9858693480491638)
 - (0.030689872801303864

In [9]:
import os
import re
import shutil

User = os.environ.get('USERNAME')  # For Windows
Year = 1942

# Your base directory containing the year and page folders
base_dir = f'C:/Users/{User}/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/{Year}'

# Directory on Google Drive where you'll upload the images
gdrive_base_dir = f'G:/My Drive/Tokyo_Jobs/{Year}'

# Iterate over each folder in the base directory
for folder_name in os.listdir(base_dir):
    # Match folders that start with 'Page' followed by numbers
    if re.match(r'Page\d+', folder_name):
        # Construct the local and destination paths
        local_img_dir = os.path.join(base_dir, folder_name, 'img')
        gdrive_img_dir = os.path.join(gdrive_base_dir, folder_name, 'img')

        # Ensure the destination directory exists
        os.makedirs(gdrive_img_dir, exist_ok=True)

        # Check if the local image directory exists and is a directory
        if os.path.isdir(local_img_dir):
            # Copy each image file from the local directory to the destination directory
            for file_name in os.listdir(local_img_dir):
                local_file_path = os.path.join(local_img_dir, file_name)
                if os.path.isfile(local_file_path):
                    dest_file_path = os.path.join(gdrive_img_dir, file_name)
                    shutil.copy(local_file_path, dest_file_path)

print("All files reorganized successfully.")

All files reorganized successfully.


In [98]:
from google.cloud import vision
from google.oauth2 import service_account
import os, json
import pandas as pd

# Add your analyze_line_layout and group_bounding_boxes functions here

def analyze_text(image_path, key_path):
    # Set the environment variable for authentication
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path

    # Create a client using the service account key
    credentials = service_account.Credentials.from_service_account_file(key_path)
    client = vision.ImageAnnotatorClient(credentials=credentials)

    with open(image_path, 'rb') as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.text_detection(image=image)

    return response.text_annotations

def split_vertical_text(text_annotations):
    new_annotations = []

    for annotation in text_annotations[1:]:  # Skipping the first annotation for the entire image
        text = annotation.description
        vertices = annotation.bounding_poly.vertices
        box_height = max(v.y for v in vertices) - min(v.y for v in vertices)
        box_width = max(v.x for v in vertices) - min(v.x for v in vertices)

        # Check for vertical text (two characters and height significantly greater than width)
        if len(text) == 2 and box_height > 2 * box_width:  # The factor 2 is adjustable based on your observations
            char_height = box_height / len(text)
            top_y = min(v.y for v in vertices)
            bottom_y = max(v.y for v in vertices)
            left_x = min(v.x for v in vertices)
            right_x = max(v.x for v in vertices)

            for i, char in enumerate(text):
                # Calculate new bounding box for each character
                new_vertices = [
                    vision.Vertex(x=left_x, y=top_y + int(i * char_height)),
                    vision.Vertex(x=right_x, y=top_y + int(i * char_height)),
                    vision.Vertex(x=right_x, y=top_y + int((i + 1) * char_height)),
                    vision.Vertex(x=left_x, y=top_y + int((i + 1) * char_height))
                ]
                new_annotation = vision.EntityAnnotation(description=char, bounding_poly=vision.BoundingPoly(vertices=new_vertices))
                new_annotations.append(new_annotation)
        else:
            new_annotations.append(annotation)

    return new_annotations


def calculate_spaces(line_boxes, text_annotations):
    # Function to find the line index for a text box
    def find_line_index_for_text_box(text_box):
        for i, line_box in enumerate(line_boxes):
            if line_box[0][0] <= text_box[0][0] <= line_box[1][0] and \
               line_box[0][1] <= text_box[0][1] <= line_box[1][1]:
                return i
        return -1

    # Organize texts by lines
    texts_by_lines = {i: [] for i in range(len(line_boxes))}
    for annotation in text_annotations[1:]:
        text = annotation.description
        box = [(v.x, v.y) for v in annotation.bounding_poly.vertices]
        line_index = find_line_index_for_text_box(box)
        if line_index != -1:
            texts_by_lines[line_index].append((text, box))

    # Sort texts within each line by x-axis and calculate spaces
    spaces_data = []
    for line_index, texts in texts_by_lines.items():
        texts.sort(key=lambda item: item[1][0][0])  # Sort by leftmost x-coordinate of bounding box
        for i in range(len(texts) - 1):
            right_edge_of_left_text = texts[i][1][1][0]
            left_edge_of_right_text = texts[i + 1][1][0][0]
            space_length = left_edge_of_right_text - right_edge_of_left_text
            spaces_data.append({
                'left_text': texts[i][0],
                'right_text': texts[i + 1][0],
                'space_length': space_length,
                'line_index': line_index  # Include the line index
            })

    return spaces_data


def detect_lines_by_right_edge(text_annotations, image_width, line_tolerance=10):
    # Extract bounding boxes and texts
    texts_with_boxes = [(anno.description, [(v.x, v.y) for v in anno.bounding_poly.vertices])
                        for anno in text_annotations[1:]]  # Skip the first annotation

    # Filter texts near the right edge
    right_edge_texts = [text for text, box in texts_with_boxes if max(v[0] for v in box) >= image_width - line_tolerance]

    # Group texts into lines
    line_groups = []
    for right_edge_text in right_edge_texts:
        _, right_edge_box = next((text, box) for text, box in texts_with_boxes if text == right_edge_text)
        current_line = [(right_edge_text, right_edge_box)]
        current_y = sum([v[1] for v in right_edge_box]) / len(right_edge_box)  # Average y-coordinate of the rightmost text

        # Trace leftward and group texts
        for text, box in texts_with_boxes:
            if (text, box) not in current_line:
                box_center_y = sum([v[1] for v in box]) / len(box)
                if abs(box_center_y - current_y) <= line_tolerance:  # Check if the text is on the same line
                    current_line.append((text, box))

        line_groups.append(current_line)

    # Sort items within each line by x-axis
    for line in line_groups:
        line.sort(key=lambda item: min(v[0] for v in item[1]))  # Sort by leftmost x-coordinate of bounding box

    # Sort lines by y-axis position of the first item in each line
    line_groups.sort(key=lambda line: sum([v[1] for v in line[0][1]]) / len(line[0][1]))  # Sort by average y-coordinate of first item

    # Extract only text from sorted lines
    sorted_text_lines = [[text for text, _ in line] for line in line_groups]

    return sorted_text_lines



def sort_line_boxes_by_y(line_boxes):
    # Filter out any boxes that are not properly structured
    valid_line_boxes = [box for box in line_boxes if len(box) == 2 and len(box[0]) == 2 and len(box[1]) == 2]

    # Sort line boxes based on the y-coordinate of the top-left corner
    sorted_line_boxes = sorted(valid_line_boxes, key=lambda box: box[0][1])
    return sorted_line_boxes

def draw_bounding_boxes(image_path, line_boxes):
    # Load the image
    image = cv2.imread(image_path)

    # Draw each bounding box
    for box in line_boxes:
        start_point, end_point = box
        image = cv2.rectangle(image, start_point, end_point, (0, 255, 0), 2)

    return image
def draw_text_on_image(image_path, text_annotations):
    # Load the image
    image = cv2.imread(image_path)

    # Draw bounding box and text for each annotation
    for annotation in text_annotations[1:]:  # Skip the first annotation which is for the entire image text
        vertices = [(v.x, v.y) for v in annotation.bounding_poly.vertices]
        text = annotation.description

        # Draw bounding box
        cv2.polylines(image, [np.array(vertices, np.int32)], True, (0, 255, 0), 2)

        # Put text
        cv2.putText(image, text, (vertices[0][0], vertices[0][1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    return image

In [99]:
image_path = 'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1942/Page/left_top.jpg'
image = Image.open(image_path)

text_annotations = analyze_text(image_path, key_path)
modified_annotations = split_vertical_text(text_annotations)
image_width,image_height = image.size

line_groups = detect_lines_by_right_edge(modified_annotations, image_width)
# Sort the line boxes based on their y-axis location
line_boxes = sort_line_boxes_by_y(line_groups)

In [100]:
line_groups

[['安', '瀨', '齊', '齊', '雄', '一', '箕', '輪', '信', '璣'],
 ['*', '根', '本', '忠', '夫', '月', '九五', '森岡', '一', '一', '光'],
 ['*', '根', '本', '忠', '夫', '月', '九五', '森岡', '一', '一', '光'],
 ['*', '根', '本', '忠', '夫', '月', '九五', '森岡', '一', '一', '光'],
 ['道', '泰', '次'],
 ['*', '下', '合', '樫', '雄', '月', '八五', '田', '直', '弘'],
 ['*', '下', '合', '樫', '雄', '月', '八五', '田', '直', '弘'],
 ['老', '上', '富', '使', '末', '勇', '月', '七五', '深', '草', '草', '元', '元', '明'],
 ['雇', '渡邊', '啓三郎', '——', '鄭', '能勢', '恒伴', '西村', '勝男'],
 ['課長', '吉', '田', '要', '助'],
 ['村松', '竹太郎'],
 ['務', '山', '岸', '祐']]

In [55]:
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import json

def extract_xml_from_html(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    return soup

def extract_coco_data(soup):
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": []
    }

    # Assuming only one page per file
    page = soup.find('page')
    image_id = 1  # or generate dynamically
    image_info = {
        "file_name": page['imagename'],
        "height": int(page['height']),
        "width": int(page['width']),
        "id": image_id
    }
    coco_data["images"].append(image_info)

    # Extract annotations
    for i, line in enumerate(page.find_all('line')):
        # Extract attributes from line
        x, y, width, height = map(int, [line.get('x'), line.get('y'), line.get('width'), line.get('height')])
        annotation = {
            "id": i,
            "image_id": image_id,
            "bbox": [x, y, width, height],
            "area": width * height,
            "iscrowd": 0
        }
        coco_data["annotations"].append(annotation)

    return coco_data


def filter_vertical_annotations(coco_annotations, ratio_threshold=1.5):
    """
    Filter out horizontal annotations from COCO annotations.

    :param coco_annotations: The COCO annotations dictionary.
    :param ratio_threshold: The minimum ratio of height to width to consider an annotation as vertical.
    :return: Filtered COCO annotations.
    """
    filtered_annotations = []
    
    for annotation in coco_annotations['annotations']:
        # In COCO format, the bounding box is [x, y, width, height]
        bbox = annotation['bbox']
        width, height = bbox[2], bbox[3]

        # Check if the height is significantly greater than the width
        if height / width > ratio_threshold:
            filtered_annotations.append(annotation)
            print(annotation)

    # Update the annotations in the COCO data
    coco_annotations['annotations'] = filtered_annotations
    return coco_annotations


In [56]:
with open('G:/My Drive/ndlocr_v2/output/image_local_2023-11-07T06 16 05.748025+09 00/b2683bda-7ce9-11ee-af13-0242ac1c000c/xml/b2683bda-7ce9-11ee-af13-0242ac1c000c.sorted.xml', 'r', encoding='utf-8') as file:
    html_content = file.read()

# Extract XML from HTML
soup = extract_xml_from_html(html_content)

# Convert to COCO format
coco_annotations = extract_coco_data(soup)
coco_annotations = filter_vertical_annotations(coco_annotations, ratio_threshold=1.5)

{'id': 0, 'image_id': 1, 'bbox': [2462, 847, 30, 229], 'area': 6870, 'iscrowd': 0}
{'id': 1, 'image_id': 1, 'bbox': [2435, 968, 21, 135], 'area': 2835, 'iscrowd': 0}
{'id': 2, 'image_id': 1, 'bbox': [2397, 759, 28, 412], 'area': 11536, 'iscrowd': 0}
{'id': 3, 'image_id': 1, 'bbox': [2370, 969, 22, 200], 'area': 4400, 'iscrowd': 0}
{'id': 4, 'image_id': 1, 'bbox': [2332, 842, 29, 335], 'area': 9715, 'iscrowd': 0}
{'id': 5, 'image_id': 1, 'bbox': [2304, 969, 22, 193], 'area': 4246, 'iscrowd': 0}
{'id': 6, 'image_id': 1, 'bbox': [2331, 423, 30, 741], 'area': 22230, 'iscrowd': 0}
{'id': 7, 'image_id': 1, 'bbox': [2471, 1794, 30, 164], 'area': 4920, 'iscrowd': 0}
{'id': 8, 'image_id': 1, 'bbox': [2467, 1218, 29, 324], 'area': 9396, 'iscrowd': 0}
{'id': 9, 'image_id': 1, 'bbox': [2429, 1219, 33, 740], 'area': 24420, 'iscrowd': 0}
{'id': 10, 'image_id': 1, 'bbox': [2377, 1222, 33, 186], 'area': 6138, 'iscrowd': 0}
{'id': 11, 'image_id': 1, 'bbox': [2328, 1219, 29, 62], 'area': 1798, 'iscrowd'

C:\Users\Keitaro Ninomiya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\bs4\builder\__init__.py:545: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  warnings.warn(


In [59]:
from PIL import Image, ImageDraw

# Load the image
image_path = 'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/HQ/Page010.jpg'  # Replace with the path to your image
image = Image.open(image_path)

# Create a drawing context
draw = ImageDraw.Draw(image)

# Iterate through all annotations and draw the bounding boxes
for annotation in coco_annotations['annotations']:
    # Extract the bounding box coordinates
    bbox = annotation['bbox']
    # Calculate the bottom-right corner of the bounding box
    x_min, y_min, width, height = bbox
    top_left = (x_min, y_min)
    bottom_right = (x_min + width, y_min + height)

    # Draw the bounding box
    draw.rectangle([top_left, bottom_right], outline="green", width=2)

# Save or display the image
annotated_image_path = 'C:/Users/Keitaro Ninomiya/Desktop/annotated_image.jpg'  # Replace with where you want to save the annotated image
image.save(annotated_image_path)
image.show()